In [1]:
import requests
import os
import pandas as pd
TOKEN_CANVAS = os.environ['TOKEN_CANVAS']

headers = {
    'Authorization': f'Bearer {TOKEN_CANVAS}',
}
params = {
    'per_page': 100,
}

In [5]:
def finn_rel(link_header):
    link_header_dict = {}
    for link in link_header.split(","):
        url, rel = link.strip().split(";")
        rel = rel.split('=')[1]
        link_header_dict[rel.strip().replace('"', '')] = url.strip().replace('<', '').replace('>', '')      # Kommentar: denne er med BERRE for å skape balanse i .org-fila: \"
    return link_header_dict

In [14]:
emne = 32028
studentar = [80292,49349,79313,93231,93970,82121,94049]

In [17]:
url = f"https://hvl.instructure.com/api/v1/courses/{emne}/enrollments?type[]=StudentEnrollment"
dr_liste = []
hentmeir = True
while hentmeir:
    respons = requests.get(url, headers=headers, params=params)
    if 200 <= respons.status_code < 300:
        data = respons.json()
        hentmeir = "next" in respons.headers['link']
        if hentmeir:
            url = finn_rel(respons.headers['link'])['next']
            print(url)   # for kontrollen sin skuld
        dataliste = []
        for element in data:
            dataliste.append([element['id'], element['user_id']])
        df = pd.DataFrame(dataliste, columns=['id', 'user_id'])
        dr_liste.append(df)
    else:
        print(f"Feil ved henting av data, responskode: {respons.status_code}")
        hentmeir = False
alledata = pd.concat((df for df in dr_liste if not df.empty), ignore_index=True)



https://hvl.instructure.com/api/v1/courses/32028/enrollments?type%5B%5D=StudentEnrollment&page=bookmark:WyJTdHVkZW50RW5yb2xsbWVudCIsIkd1c3RhdnNlbiwgVG9tYXMiLDE1MzUzNTRd&per_page=100
https://hvl.instructure.com/api/v1/courses/32028/enrollments?type%5B%5D=StudentEnrollment&page=bookmark:WyJTdHVkZW50RW5yb2xsbWVudCIsIktodXUsIEVtaWwiLDE2MTQyODdd&per_page=100
https://hvl.instructure.com/api/v1/courses/32028/enrollments?type%5B%5D=StudentEnrollment&page=bookmark:WyJTdHVkZW50RW5yb2xsbWVudCIsIlNhbG9tb25zZW4sIMOYcmphbiIsMTYxNDA4M10&per_page=100


In [21]:
v = alledata[alledata['user_id']==94049]['id'].values[0]
print(v)

1614110


In [24]:
for l in studentar:
    if l in alledata['user_id'].values:
        v = alledata[alledata['user_id']==94049]['id'].values[0]
        url = f"https://hvl.instructure.com/api/v1/courses/{emne}/enrollments/{v}?task=conclude"
        respons2 = requests.delete(url, headers=headers)
        print(f"Brukar {l} avslutta, responskode: {respons2.status_code}")
    else:
        print(f"Brukar {l} ikkje på lista, ingen endring.")


Brukar 80292 ikkje på lista, ingen endring.
Brukar 49349 avslutta, responskode: 200
Brukar 79313 avslutta, responskode: 200
Brukar 93231 avslutta, responskode: 200
Brukar 93970 avslutta, responskode: 200
Brukar 82121 avslutta, responskode: 200
Brukar 94049 avslutta, responskode: 200
